# Fakes and Dependency Overrides

This notebook covers:

1. The `app.dependency_overrides` dict — what it is, how FastAPI uses it, and why it's the cleanest seam for tests
2. Fakes vs mocks vs the real backend — when each is right, and the cost of each
3. Replacing a SQLite repository with an in-memory fake **per test**
4. The `yield` + cleanup pattern that guarantees overrides don't leak between tests
5. A fake clock for time-sensitive tests (auth token expiry, rate-limit windows)

**Scope**: FastAPI + Pydantic v2 + pytest. Builds on the `AssetRepo` Protocol pattern from notebook 4.2 — same shape, exercised from a test seat. All test files run via `python -m pytest` so the output is the real CLI output.

## 1. `app.dependency_overrides`

FastAPI's DI graph (notebooks 4.1–4.3) is what makes the framework testable. Every `Depends(get_thing)` is, mechanically, a callable that FastAPI invokes to fill a parameter. Tests need to swap those callables — to use an in-memory store instead of Postgres, a fake clock instead of `datetime.now`, a stub of the email-sender instead of SMTP.

`app.dependency_overrides` is the seam. It's a dict on the app: `{ original_callable: replacement_callable }`. When FastAPI is about to invoke `Depends(original)`, it checks the dict; if there's an entry, it calls the replacement instead. That's it. No metaclass magic, no monkeypatch, no module reload.

Three properties worth memorizing:

- **The key is the *original* callable.** `app.dependency_overrides[get_repo] = lambda: fake_repo`. The replacement gets called with the same arguments — usually none, if `get_repo` is a zero-arg provider.
- **It's scoped to the app object.** Different apps don't share overrides. If you have multiple `FastAPI()` instances in a test suite, override on each.
- **It persists across tests until you clear it.** If `test_a` writes an override and `test_b` doesn't reset it, `test_b` runs against the fake. That's the failure mode section 4 exists to prevent.

In [1]:
# The minimal demonstration: one dependency, one override, one test.
from fastapi import Depends, FastAPI
from fastapi.testclient import TestClient

def get_greeting() -> str:
    # Pretend this hits a config file or an external service.
    return "hello from production"

app = FastAPI()

@app.get("/")
def root(greeting: str = Depends(get_greeting)):
    return {"greeting": greeting}

# Live: the real provider runs.
print("live :", TestClient(app).get("/").json())

# Override: the test substitutes a constant.
app.dependency_overrides[get_greeting] = lambda: "hello from a test"
print("test :", TestClient(app).get("/").json())

# Clean up so the next cell isn't tainted.
app.dependency_overrides.clear()
print("clean:", TestClient(app).get("/").json())

live : {'greeting': 'hello from production'}
test : {'greeting': 'hello from a test'}
clean: {'greeting': 'hello from production'}


C:\Users\mathi\AppData\Roaming\Python\Python314\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


That `clear()` call is the discipline — and the only thing standing between you and a flaky suite. Section 4 will package it inside a fixture so you never forget.

## 2. Fakes vs Mocks vs the Real Backend

Before writing one more line of test code, get the vocabulary right — these words get mixed up and the choice between them is the single biggest determinant of how good a test suite feels to use.

- **Fake**: a *working* implementation of the same interface, optimized for tests. An in-memory dict-backed repository is a fake of a SQLite repository: it satisfies the same `Protocol`, returns the same types, raises the same exceptions — it just doesn't persist. Fakes are honest; if you write a `.save()` then `.get()` on a fake, you get back what you saved.
- **Mock**: a stand-in whose behavior is configured per-test ("when `.get('AAPL')` is called, return this dict"). `unittest.mock.Mock` and `MagicMock` are the canonical Python implementations. Mocks are *dishonest by default* — they answer however you told them to, which is great for asserting "was this called?" and bad for asserting "does the system still work?"
- **Real backend**: the actual SQLite/Postgres/Redis/HTTP service. Most truthful, slowest, most setup. The integration tier.

A useful rule of thumb:

- **Fakes are the default.** They give you almost all the truth of an integration test for almost the speed of a unit test. Build a fake repo once, use it across the whole suite.
- **Mocks are for asserting interactions.** "Did the route call `email_service.send()` exactly once with these args?" — that's a mock job. Don't use a mock just because "I needed a thing to return a value"; that's what a fake is for.
- **Real backends are for the integration tier.** A handful of tests that prove the SQL migration actually runs, that JWTs decode against the real key. Run them in CI; do not run them on every save.

The cost of getting this wrong is concrete: mock-heavy suites pass while the system is broken because the mocks were updated alongside the broken code. Fake-based suites tend to fail loudly when the interface drifts, because the fake has to keep satisfying the same contract.

## 3. Building a Fake Repo and Wiring It In

Notebook 4.2 introduced the `AssetRepo` Protocol with `InMemoryAssetRepo` and `SQLiteAssetRepo`. The exact same Protocol shape is what we override for tests — the production app gets the SQLite repo, the test gets the in-memory fake, neither code path knows.

Below we set the whole thing up in one cell so the rest of the notebook can refer to it. The app's route depends on `get_repo()`; the test overrides `get_repo()` to return a fake. The route is unchanged.

In [2]:
from typing import Protocol
from fastapi import Depends, FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

class Asset(BaseModel):
    ticker: str = Field(pattern=r"^[A-Z.]{1,10}$")
    name: str = Field(min_length=1)
    price: float = Field(ge=0)

class AssetRepo(Protocol):
    def get(self, ticker: str) -> Asset | None: ...
    def add(self, asset: Asset) -> None: ...
    def list(self) -> list[Asset]: ...

class InMemoryAssetRepo:
    # The fake. Same interface as the SQLite repo would have, no persistence.
    def __init__(self, seed: list[Asset] | None = None):
        self._data: dict[str, Asset] = {a.ticker: a for a in (seed or [])}
    def get(self, ticker): return self._data.get(ticker.upper())
    def add(self, asset): self._data[asset.ticker] = asset
    def list(self): return list(self._data.values())

# The provider — in real life this would open a SQLite or Postgres connection.
# The test will replace this entire function.
_PRODUCTION_REPO = InMemoryAssetRepo([Asset(ticker="AAPL", name="Apple Inc.", price=190.0)])
def get_repo() -> AssetRepo:
    return _PRODUCTION_REPO

app = FastAPI()

@app.get("/assets/{ticker}", response_model=Asset)
def read_asset(ticker: str, repo: AssetRepo = Depends(get_repo)):
    asset = repo.get(ticker)
    if asset is None:
        raise HTTPException(404, f"Asset '{ticker}' not found")
    return asset

@app.get("/assets", response_model=list[Asset])
def list_assets(repo: AssetRepo = Depends(get_repo)):
    return repo.list()

# Live: the "production" provider returns its seeded data.
print("live AAPL :", TestClient(app).get("/assets/AAPL").json())
print("live list :", TestClient(app).get("/assets").json())

live AAPL : {'ticker': 'AAPL', 'name': 'Apple Inc.', 'price': 190.0}
live list : [{'ticker': 'AAPL', 'name': 'Apple Inc.', 'price': 190.0}]


In [3]:
# Test seat: override get_repo to return a *different* fake with different data.
test_repo = InMemoryAssetRepo([
    Asset(ticker="TSLA", name="Tesla Inc.", price=250.0),
    Asset(ticker="NVDA", name="NVIDIA Corp.", price=900.0),
])
app.dependency_overrides[get_repo] = lambda: test_repo

c = TestClient(app)
print("test list :", c.get("/assets").json())
print("test TSLA :", c.get("/assets/TSLA").json())
# AAPL was in the live repo, not the test repo — proves the override is in effect.
print("test AAPL :", c.get("/assets/AAPL").status_code, c.get("/assets/AAPL").json())

app.dependency_overrides.clear()

test list : [{'ticker': 'TSLA', 'name': 'Tesla Inc.', 'price': 250.0}, {'ticker': 'NVDA', 'name': 'NVIDIA Corp.', 'price': 900.0}]
test TSLA : {'ticker': 'TSLA', 'name': 'Tesla Inc.', 'price': 250.0}
test AAPL : 404 {'detail': "Asset 'AAPL' not found"}


Notice what didn't change: the route function. There is no `if testing:` branch, no special test-mode environment variable, no module reloading. The seam is the `Depends(get_repo)` boundary; the override pivots one half without touching the other. That's the entire payoff of designing your app around DI in the first place.

## 4. Per-Test Overrides — and Cleaning Them Up

The smoke-test cells above called `app.dependency_overrides.clear()` manually. In a real suite of 200 tests, *someone* will forget. The fix is to package set + cleanup in a fixture so pytest does it for you.

The pattern below is the canonical shape. Notice three things:

- The fixture **yields**, then runs cleanup after the test returns. That cleanup runs even if the test failed.
- Cleanup `.clear()`s overrides for *only the keys the fixture added*, so multiple fixtures composing overrides don't stomp on each other.
- The fixture **returns the fake**, so the test can introspect it (e.g., "did the route's POST actually call `.add()` on the fake?").

In [4]:
import subprocess, sys
from pathlib import Path
from tempfile import mkdtemp
from textwrap import dedent

def run_pytest(files: dict[str, str], *args: str) -> str:
    d = Path(mkdtemp(prefix="pytest_overrides_"))
    for name, src in files.items():
        (d / name).write_text(dedent(src).lstrip(), encoding="utf-8")
    result = subprocess.run(
        [sys.executable, "-m", "pytest", str(d), "-q", "--no-header", *args],
        capture_output=True, text=True,
    )
    return result.stdout + result.stderr

APP_PY = '''
from typing import Protocol
from fastapi import Depends, FastAPI, HTTPException
from pydantic import BaseModel, Field

class Asset(BaseModel):
    ticker: str = Field(pattern=r"^[A-Z.]{1,10}$")
    name: str = Field(min_length=1)
    price: float = Field(ge=0)

class AssetRepo(Protocol):
    def get(self, ticker: str) -> Asset | None: ...
    def add(self, asset: Asset) -> None: ...
    def list(self) -> list[Asset]: ...

class InMemoryAssetRepo:
    def __init__(self, seed=None):
        self._data: dict[str, Asset] = {a.ticker: a for a in (seed or [])}
    def get(self, ticker): return self._data.get(ticker.upper())
    def add(self, asset): self._data[asset.ticker] = asset
    def list(self): return list(self._data.values())

def get_repo() -> AssetRepo:
    # Production wiring would build a SQLite / Postgres repo here.
    raise RuntimeError("Tests must override get_repo")

app = FastAPI()

@app.get("/assets", response_model=list[Asset])
def list_assets(repo: AssetRepo = Depends(get_repo)):
    return repo.list()

@app.get("/assets/{ticker}", response_model=Asset)
def read_asset(ticker: str, repo: AssetRepo = Depends(get_repo)):
    a = repo.get(ticker)
    if a is None:
        raise HTTPException(404, f"Asset '{ticker}' not found")
    return a

@app.post("/assets", response_model=Asset, status_code=201)
def create_asset(asset: Asset, repo: AssetRepo = Depends(get_repo)):
    if repo.get(asset.ticker) is not None:
        raise HTTPException(409, f"Ticker '{asset.ticker}' exists")
    repo.add(asset)
    return asset
'''

CONFTEST = '''
import pytest
from fastapi.testclient import TestClient
from app import app, get_repo, InMemoryAssetRepo, Asset

@pytest.fixture
def repo():
    # A fresh in-memory fake per test — total isolation.
    return InMemoryAssetRepo()

@pytest.fixture
def client(repo):
    # Set the override, hand out the client, then clean up afterwards.
    # try/finally guarantees the cleanup even if the test raises.
    app.dependency_overrides[get_repo] = lambda: repo
    try:
        yield TestClient(app)
    finally:
        app.dependency_overrides.pop(get_repo, None)
'''

TEST_OVERRIDE = '''
from app import Asset

def test_get_unknown_returns_404(client):
    assert client.get("/assets/AAPL").status_code == 404

def test_create_then_get(client):
    payload = {"ticker": "AAPL", "name": "Apple Inc.", "price": 190.0}
    assert client.post("/assets", json=payload).status_code == 201
    assert client.get("/assets/AAPL").json() == payload

def test_isolation_between_tests(client, repo):
    # If the previous test's override or repo leaked, AAPL would still be here.
    # It isn't, because both `repo` and the override are function-scoped.
    assert repo.list() == []
    assert client.get("/assets").json() == []

def test_fixture_introspection(client, repo):
    # Returning the fake from the fixture lets the test assert on side-effects
    # without going through HTTP a second time.
    client.post("/assets", json={"ticker": "MSFT", "name": "Microsoft", "price": 420.0})
    assert len(repo.list()) == 1
    assert repo.get("MSFT").price == 420.0
'''

output = run_pytest({"app.py": APP_PY, "conftest.py": CONFTEST, "test_override.py": TEST_OVERRIDE}, "-v")
print(output)

============================= test session starts =============================
collected 4 items

..\..\..\..\AppData\Local\Temp\pytest_overrides_zc3vqynf\test_override.py . [ 25%]
...                                                                      [100%]

============================== warnings summary ===============================
..\..\..\..\AppData\Roaming\Python\Python314\site-packages\fastapi\testclient.py:1
  C:\Users\mathi\AppData\Roaming\Python\Python314\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
    from starlette.testclient import TestClient as TestClient  # noqa

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
======================== 4 passed, 1 warning in 0.55s =========================



Two design notes worth reading even if all four tests passed:

- **`.pop(get_repo, None)` beats `.clear()`** when fixtures might compose. If three fixtures each add an override and they all call `.clear()`, fixture A's cleanup wipes fixture B's override mid-test. Removing only your own key is the polite contract.
- **The fixture returning the `repo`** is a quiet but powerful pattern. It lets the test poke at internal state — "did the call actually persist?", "is the queue empty after the request?" — without needing a second HTTP round trip to find out. That's a kind of white-box testing only an in-process test seat can do.

## 5. A Fake Clock for Time-Sensitive Tests

Time is the dependency people forget to inject. Auth tokens have an `exp` claim (notebook 5.2). Rate limiters have a window (5.3). Caches have a TTL. Most teams hit one of these patterns by writing `time.sleep(2)` in the test — slow, flaky, infuriating.

The clean version: make `now()` a dependency, and override it with a fake.

In [5]:
from datetime import datetime, timezone, timedelta
from fastapi import Depends, FastAPI
from fastapi.testclient import TestClient

# The clock dependency. In production it's `datetime.now`; in tests it's a fake.
def get_now() -> datetime:
    return datetime.now(timezone.utc)

# Imagine a token issued at T with a 10-minute lifetime.
TOKEN_LIFETIME = timedelta(minutes=10)
ISSUED_AT = datetime(2026, 1, 1, 12, 0, 0, tzinfo=timezone.utc)

app2 = FastAPI()

@app2.get("/token-status")
def token_status(now: datetime = Depends(get_now)):
    age = now - ISSUED_AT
    return {"age_seconds": int(age.total_seconds()), "expired": age > TOKEN_LIFETIME}

class FakeClock:
    """Mutable fake the test drives manually. tick() advances simulated time."""
    def __init__(self, start: datetime):
        self.now = start
    def __call__(self) -> datetime:
        return self.now
    def tick(self, seconds: float) -> None:
        self.now += timedelta(seconds=seconds)

clock = FakeClock(ISSUED_AT + timedelta(minutes=5))
app2.dependency_overrides[get_now] = clock  # the clock IS the callable

c = TestClient(app2)
print("at +5min  :", c.get("/token-status").json())

clock.tick(seconds=400)  # advance to +5min + 6m 40s = +11min 40s
print("at +11:40 :", c.get("/token-status").json())

app2.dependency_overrides.clear()

at +5min  : {'age_seconds': 300, 'expired': False}
at +11:40 : {'age_seconds': 700, 'expired': True}


Two observations to internalize:

- **The test fired three requests and never slept.** What would have been a 10-minute integration test runs in microseconds because *the test owns the clock*. The same pattern works for rate-limit windows, JWT expiry, scheduler windows, anything time-driven.
- **The fake clock is the override callable itself.** `clock` is callable (`__call__`), so `app.dependency_overrides[get_now] = clock` works directly — no `lambda: clock`. That's a small ergonomic win and a hint about how flexible the override mechanism is.

Production code that does `datetime.now()` inline (no dependency) is *untestable in time*. Pushing time through a `get_now` dep is the kind of small architectural choice that pays off in a half-dozen tests' worth of friction saved.

## Key Takeaways

- **`app.dependency_overrides[orig] = replacement`** is the entire DI test seat. No reloads, no monkeypatch, no env-var gymnastics — and it works precisely because notebooks 4.1–4.3 routed everything through `Depends`.
- **Fakes are the default; mocks are for interactions; real backends are the integration tier.** Misclassifying these is the most common reason test suites either lie or run too slowly.
- **A function-scoped fixture that sets the override, yields the client, and `.pop()`s the override in `finally`** is the canonical isolation pattern. Use `.pop(key, None)` rather than `.clear()` so composed fixtures don't stomp each other.
- **Returning the fake from the fixture** is what enables white-box assertions — checking the in-memory state directly instead of making a second HTTP call to observe a side effect.
- **Inject time.** A `get_now` dependency + a `FakeClock` overrideable callable replaces `time.sleep(N)` in token-expiry, rate-limit, and TTL tests — millisecond tests that exercise minute-long behavior.
- **Capstone tie-in**: the capstone's `tests/conftest.py` will use exactly this pattern — an `InMemoryAssetRepo` fake registered against `get_repo`, plus a `FakeClock` registered against the auth module's `get_now`. The production app code remains untouched.

## Exercises

All exercises use the `run_pytest({...})` helper from section 4 — write the file dict, run pytest, read the output.

**1. Test isolation under failure.** Add a test `test_failing_test_does_not_leak_override` that asserts the fixture's override is gone after a prior test deliberately fails. The trick: use a *new* fixture that records `app.dependency_overrides` after the test, and a test that raises. (Hint: pytest's `request.node.add_report_section` or simply look up `dependency_overrides` from a separate teardown step.) The goal is to confirm the `try/finally` actually fires on test failure.

**2. A failing fake for resilience tests.** Build a second fake `FailingAssetRepo` whose `get()` always raises `RuntimeError("db down")`. Write a test that overrides `get_repo` with this failing fake, calls `GET /assets/AAPL`, and asserts the response is a 500 with the standard error envelope (or whatever your handler chain produces). This is how you verify the *unhappy path* of your error-handler stack from notebook 6.1.

**3. Fake clock + rate limit.** Build a tiny rate-limiter dependency: `get_rate_limiter` returns a callable that allows N requests per 60-second window per IP, using the `get_now` clock dependency. Write a test that fires N+1 requests against a route that uses the limiter, asserts the (N+1)th returns 429, then `clock.tick(seconds=61)` and asserts the next request returns 200. The whole test should run in microseconds — that's the point of injecting time.